In [10]:
HAPTIC_COST = {
    0: 0.00,
    1: 0.03,
    2: 0.08
}

In [11]:
reward_configs = {
    "Low_penalty": {
        "improvement_weight": 2.0,
        "costs": {0: 0.00, 1: 0.01, 2: 0.03}
    },

    "Current": {
        "improvement_weight": 2.0,
        "costs": {0: 0.00, 1: 0.03, 2: 0.08}
    },

    "High_penalty": {
        "improvement_weight": 2.0,
        "costs": {0: 0.00, 1: 0.05, 2: 0.12}
    }
}

In [12]:
def calculate_reward_configurable(
    correct,
    p_before,
    p_after,
    action,
    improvement_weight,
    costs
):
    success_reward = 1.0 if correct == 1 else -0.5

    improvement = p_after - p_before

    reward = (
        success_reward
        + improvement_weight * improvement
        - costs[action]
    )

    return reward

In [13]:
def environment_step_configurable(
    row,
    performance,
    action,
    rng,
    improvement_weight,
    costs
):
    ability = float(row["Learner_Ability"])
    difficulty = float(row["Difficulty_D"])
    cognitive_effort = float(row["Cognitive_Effort_C"])

    p_before = probability_correct(
        ability=ability,
        difficulty=difficulty,
        cognitive_effort=cognitive_effort
    )

    p_after = probability_correct_with_haptics(
        ability=ability,
        difficulty=difficulty,
        cognitive_effort=cognitive_effort,
        performance=performance,
        action=action
    )

    p_after = np.clip(p_after, 0.0, 1.0)

    correct = rng.binomial(
        n=1,
        p=p_after
    )

    reward = calculate_reward_configurable(
        correct=correct,
        p_before=p_before,
        p_after=p_after,
        action=action,
        improvement_weight=improvement_weight,
        costs=costs
    )

    new_performance = (
        0.8 * performance
        + 0.2 * correct
    )

    new_performance = np.clip(
        new_performance,
        0.0,
        1.0
    )

    return (
        new_performance,
        reward,
        correct,
        p_before,
        p_after
    )

In [5]:
def train_qlearning(
    simulation_df,
    improvement_weight,
    costs,
    seed=42,
    n_episodes=500,
    steps_per_episode=50
):
    rng = np.random.default_rng(seed)

    states = [
        (d, c, p)
        for d in range(3)
        for c in range(3)
        for p in range(3)
    ]

    Q_local = {
        state: np.zeros(3)
        for state in states
    }

    epsilon = 1.0

    for episode in range(n_episodes):

        row = simulation_df.iloc[
            rng.integers(0, len(simulation_df))
        ]

        performance = float(
            row["Performance_P"]
        )

        for step in range(steps_per_episode):

            state = (
                int(row["D_level"]),
                int(row["C_level"]),
                performance_level(performance)
            )

            action = choose_action(
                state,
                Q_local,
                epsilon,
                rng
            )

            (
                new_performance,
                reward,
                correct,
                _,
                _
            ) = environment_step_configurable(
                row=row,
                performance=performance,
                action=action,
                rng=rng,
                improvement_weight=improvement_weight,
                costs=costs
            )

            learner_rows = simulation_df[
                simulation_df["Learner_Profile"]
                == row["Learner_Profile"]
            ]

            next_row = learner_rows.iloc[
                rng.integers(
                    0,
                    len(learner_rows)
                )
            ]

            next_state = (
                int(next_row["D_level"]),
                int(next_row["C_level"]),
                performance_level(new_performance)
            )

            current_q = Q_local[state][action]
            best_next_q = np.max(Q_local[next_state])

            target = (
                reward
                + 0.95 * best_next_q
            )

            Q_local[state][action] = (
                current_q
                + 0.10 * (
                    target - current_q
                )
            )

            performance = new_performance
            row = next_row

        epsilon = max(
            0.05,
            epsilon * 0.995
        )

    policy_local = {
        state: int(np.argmax(Q_local[state]))
        for state in states
    }

    return Q_local, policy_local

In [16]:
import pandas as pd
from collections import Counter

simulation_df = pd.read_csv(
    "/Users/nataliakolbina/PycharmProjects/JupyterProject/Reproduce ReadMe+++/phase10_synthetic_learner_environment.csv"
)

print(simulation_df.shape)
print(simulation_df.columns.tolist()[:10])
simulation_df.head()

(2127, 21)
['Excerpt', 'BT.Easiness', 'Predicted_BT_Easiness', 'Difficulty_D', 'Gaze_Score_Raw', 'Cognitive_Effort_C', 'D_level', 'C_level', 'D_label', 'C_label']


,Excerpt,BT.Easiness,Predicted_BT_Easiness,Difficulty_D,Gaze_Score_Raw,Cognitive_Effort_C,D_level,C_level,D_label,C_label,...,D_level_q,C_level_q,Learner_Profile,Learner_Ability,P_correct,Correct,Performance_P,P_level,P_label,RL_State
0,"Anywhere there is a frontier, where there are ...",-1.866238,-1.335351,0.566335,-23.967209,0.376504,2,1,High,Medium,...,2,1,Average,0.55,0.437328,1,1.000000,2,High,"(2, 1, 2)"
1,"But Little Ant was extremely thirsty. ""I will ...",-0.206375,0.459791,0.232650,-42.317612,0.177729,0,0,Low,Low,...,0,0,Average,0.55,0.758888,0,0.500000,1,Medium,"(0, 0, 1)"
2,"So, what was wrong? A group of music neuroscie...",-1.610294,-0.928272,0.490666,-29.344208,0.318259,1,0,Medium,Low,...,1,0,Average,0.55,0.522855,0,0.333333,1,Medium,"(1, 0, 1)"
3,"Long, long ago, the Sky God, Nyame, kept all s...",0.227890,0.001327,0.317870,-24.188194,0.374110,0,1,Low,Medium,...,0,1,Average,0.55,0.622029,0,0.250000,0,Low,"(0, 1, 0)"
4,"Now, all these experiences of living in differ...",-0.261222,-0.090363,0.334914,-37.383026,0.231181,0,0,Low,Low,...,0,0,Average,0.55,0.675440,1,0.400000,1,Medium,"(0, 0, 1)"


In [21]:
import numpy as np
from collections import defaultdict, Counter

def performance_level(p):
    if p < 1/3:
        return 0
    elif p < 2/3:
        return 1
    else:
        return 2

def choose_action(state, Q, epsilon, rng):
    if rng.random() < epsilon:
        return int(rng.choice([0, 1, 2]))
    return int(np.argmax(Q[state]))

In [22]:
for name in [
    "np",
    "defaultdict",
    "performance_level",
    "choose_action",
    "environment_step_configurable",
    "simulation_df",
    "reward_configs",
    "train_qlearning",
]:
    print(name, name in globals())

np True
defaultdict True
performance_level True
choose_action True
environment_step_configurable True
simulation_df True
reward_configs True
train_qlearning True


In [25]:
import math

def probability_correct(
    ability,
    difficulty,
    cognitive_effort,
    alpha=3.0,
    beta=2.0,
    gamma=1.5
):
    z = (
        alpha * ability
        - beta * difficulty
        - gamma * cognitive_effort
    )
    return 1 / (1 + math.exp(-z))

In [26]:
print("probability_correct" in globals())

True


In [30]:
import math

def haptic_effect(action, difficulty, cognitive_effort, performance):
    need = (
        0.4 * difficulty
        + 0.4 * cognitive_effort
        + 0.2 * (1 - performance)
    )

    if action == 0:   # Low
        effect = 0.08 * need
    elif action == 1: # Medium
        effect = 0.18 * need
    elif action == 2: # High
        effect = 0.28 * need
    else:
        raise ValueError("action must be 0, 1, or 2")

    return effect


def probability_correct_with_haptics(
    ability,
    difficulty,
    cognitive_effort,
    performance,
    action,
    alpha=2.0,
    beta=2.0
):
    base_logit = (
        alpha * ability
        - beta * difficulty
        - beta * cognitive_effort
        + performance
    )

    boost = haptic_effect(
        action=action,
        difficulty=difficulty,
        cognitive_effort=cognitive_effort,
        performance=performance
    )

    boosted_logit = base_logit + boost

    return 1 / (1 + math.exp(-boosted_logit))

In [31]:
for name in [
    "np",
    "Counter",
    "performance_level",
    "choose_action",
    "probability_correct",
    "haptic_effect",
    "probability_correct_with_haptics",
    "environment_step_configurable",
    "train_qlearning",
    "simulation_df",
    "reward_configs"
]:
    print(name, name in globals())

np True
Counter True
performance_level True
choose_action True
probability_correct True
haptic_effect True
probability_correct_with_haptics True
environment_step_configurable True
train_qlearning True
simulation_df True
reward_configs True


In [32]:
sensitivity_Q = {}
sensitivity_policies = {}

for name, cfg in reward_configs.items():

    Q_temp, policy_temp = train_qlearning(
        simulation_df=simulation_df,
        improvement_weight=cfg["improvement_weight"],
        costs=cfg["costs"]
    )

    sensitivity_Q[name] = Q_temp
    sensitivity_policies[name] = policy_temp

    print(name, Counter(policy_temp.values()))

Low_penalty Counter({2: 13, 0: 8, 1: 6})
Current Counter({1: 13, 0: 11, 2: 3})
High_penalty Counter({0: 14, 1: 8, 2: 5})


In [33]:
def state_full(row, performance):
    return (
        int(row["D_level"]),
        int(row["C_level"]),
        performance_level(performance)
    )


def state_dp(row, performance):
    return (
        int(row["D_level"]),
        performance_level(performance)
    )


def state_d(row, performance):
    return (
        int(row["D_level"]),
    )

In [34]:
def train_qlearning_ablation(
    simulation_df,
    state_function,
    possible_states,
    seed=42,
    n_episodes=500,
    steps_per_episode=50
):
    rng = np.random.default_rng(seed)

    Q_local = {
        state: np.zeros(3)
        for state in possible_states
    }

    epsilon = 1.0

    for episode in range(n_episodes):

        row = simulation_df.iloc[
            rng.integers(0, len(simulation_df))
        ]

        performance = float(
            row["Performance_P"]
        )

        for step in range(steps_per_episode):

            state = state_function(
                row,
                performance
            )

            action = choose_action(
                state,
                Q_local,
                epsilon,
                rng
            )

            (
                new_performance,
                reward,
                correct,
                _,
                _
            ) = environment_step(
                row=row,
                performance=performance,
                action=action,
                rng=rng
            )

            learner_rows = simulation_df[
                simulation_df["Learner_Profile"]
                == row["Learner_Profile"]
            ]

            next_row = learner_rows.iloc[
                rng.integers(
                    0,
                    len(learner_rows)
                )
            ]

            next_state = state_function(
                next_row,
                new_performance
            )

            current_q = Q_local[state][action]
            best_next_q = np.max(Q_local[next_state])

            target = (
                reward
                + 0.95 * best_next_q
            )

            Q_local[state][action] = (
                current_q
                + 0.10
                * (target - current_q)
            )

            performance = new_performance
            row = next_row

        epsilon = max(
            0.05,
            epsilon * 0.995
        )

    policy_local = {
        state: int(np.argmax(Q_local[state]))
        for state in possible_states
    }

    return Q_local, policy_local

In [35]:
states_full = [
    (d, c, p)
    for d in range(3)
    for c in range(3)
    for p in range(3)
]

states_dp = [
    (d, p)
    for d in range(3)
    for p in range(3)
]

states_d = [
    (d,)
    for d in range(3)
]

In [39]:
def train_qlearning_ablation(
    simulation_df,
    state_function,
    possible_states,
    seed=123,
    n_episodes=300,
    steps_per_episode=50
):
    rng = np.random.default_rng(seed)

    Q = {
        state: np.zeros(3)
        for state in possible_states
    }

    epsilon = 1.0
    learning_rate = 0.1
    discount_factor = 0.9
    epsilon_decay = 0.995
    epsilon_min = 0.05

    for episode in range(n_episodes):

        row = simulation_df.iloc[
            rng.integers(0, len(simulation_df))
        ]

        performance = float(row["Performance_P"])

        for step in range(steps_per_episode):

            state = state_function(row, performance)

            action = choose_action(
                state,
                Q,
                epsilon,
                rng
            )

            (
                new_performance,
                reward,
                correct,
                _,
                _
            ) = environment_step_configurable(
                row=row,
                performance=performance,
                action=action,
                rng=rng,
                improvement_weight=1.0,
                costs={
                    0: 0.00,
                    1: 0.03,
                    2: 0.07
                }
            )

            learner_rows = simulation_df[
                simulation_df["Learner_Profile"]
                == row["Learner_Profile"]
            ]

            next_row = learner_rows.iloc[
                rng.integers(0, len(learner_rows))
            ]

            next_state = state_function(
                next_row,
                new_performance
            )

            old_q = Q[state][action]
            best_next_q = np.max(Q[next_state])

            target = reward + discount_factor * best_next_q

            Q[state][action] = old_q + learning_rate * (
                target - old_q
            )

            row = next_row
            performance = new_performance

        epsilon = max(
            epsilon_min,
            epsilon * epsilon_decay
        )

    policy = {
        state: int(np.argmax(Q[state]))
        for state in possible_states
    }

    return Q, policy


In [40]:
Q_full, policy_full = train_qlearning_ablation(
    simulation_df,
    state_full,
    states_full
)

Q_dp, policy_dp = train_qlearning_ablation(
    simulation_df,
    state_dp,
    states_dp
)

Q_d, policy_d = train_qlearning_ablation(
    simulation_df,
    state_d,
    states_d
)

In [41]:
from collections import Counter

print("FULL (D, C, P)")
print("Number of states:", len(policy_full))
print("Actions:", Counter(policy_full.values()))

print("\nREDUCED (D, P)")
print("Number of states:", len(policy_dp))
print("Actions:", Counter(policy_dp.values()))

print("\nDIFFICULTY ONLY (D)")
print("Number of states:", len(policy_d))
print("Actions:", Counter(policy_d.values()))

FULL (D, C, P)
Number of states: 27
Actions: Counter({1: 11, 2: 9, 0: 7})

REDUCED (D, P)
Number of states: 9
Actions: Counter({0: 6, 2: 2, 1: 1})

DIFFICULTY ONLY (D)
Number of states: 3
Actions: Counter({0: 2, 2: 1})


In [42]:
def evaluate_ablation_policy(
    policy_name,
    policy_to_use,
    state_function,
    n_episodes=300,
    steps_per_episode=50,
    seed=123
):
    rng = np.random.default_rng(seed)

    rewards = []
    accuracies = []
    action_history = []

    for episode in range(n_episodes):

        total_reward = 0
        total_correct = 0

        # Start with random learner/content row
        row_idx = rng.integers(
            0,
            len(simulation_df)
        )

        row = simulation_df.iloc[row_idx]

        performance = float(
            row["Performance_P"]
        )

        for step in range(steps_per_episode):

            # Build state using the correct representation
            state = state_function(
                row,
                performance
            )

            # IMPORTANT:
            # use the learned policy directly
            action = policy_to_use[state]

            action_history.append(action)

            (
                new_performance,
                reward,
                correct,
                p_before,
                p_after
            ) = environment_step(
                row=row,
                performance=performance,
                action=action,
                rng=rng
            )

            total_reward += reward
            total_correct += correct

            performance = new_performance

            # Keep same learner profile
            learner_rows = simulation_df[
                simulation_df["Learner_Profile"]
                == row["Learner_Profile"]
            ]

            row = learner_rows.iloc[
                rng.integers(
                    0,
                    len(learner_rows)
                )
            ]

        rewards.append(
            total_reward / steps_per_episode
        )

        accuracies.append(
            total_correct / steps_per_episode
        )

    return {
        "model": policy_name,
        "mean_reward": np.mean(rewards),
        "std_reward": np.std(rewards),
        "mean_accuracy": np.mean(accuracies),
        "std_accuracy": np.std(accuracies),
        "mean_action": np.mean(action_history),
        "low_prop": np.mean(
            np.array(action_history) == 0
        ),
        "medium_prop": np.mean(
            np.array(action_history) == 1
        ),
        "high_prop": np.mean(
            np.array(action_history) == 2
        )
    }

In [44]:
def evaluate_ablation_policy(
    policy_name,
    policy_to_use,
    state_function,
    n_episodes=300,
    steps_per_episode=50,
    seed=123
):
    rng = np.random.default_rng(seed)

    rewards = []
    accuracies = []

    for episode in range(n_episodes):

        row = simulation_df.iloc[
            rng.integers(0, len(simulation_df))
        ]

        performance = float(
            row["Performance_P"]
        )

        total_reward = 0
        total_correct = 0

        for step in range(steps_per_episode):

            state = state_function(
                row,
                performance
            )

            action = int(
                policy_to_use[state]
            )

            (
                new_performance,
                reward,
                correct,
                p_before,
                p_after
            ) = environment_step_configurable(
                row=row,
                performance=performance,
                action=action,
                rng=rng,
                improvement_weight=1.0,
                costs={
                    0: 0.00,
                    1: 0.03,
                    2: 0.07
                }
            )

            total_reward += reward
            total_correct += correct

            learner_rows = simulation_df[
                simulation_df["Learner_Profile"]
                == row["Learner_Profile"]
            ]

            row = learner_rows.iloc[
                rng.integers(0, len(learner_rows))
            ]

            performance = new_performance

        rewards.append(
            total_reward / steps_per_episode
        )

        accuracies.append(
            total_correct / steps_per_episode
        )

    return {
        "policy": policy_name,
        "mean_reward": np.mean(rewards),
        "std_reward": np.std(rewards),
        "mean_accuracy": np.mean(accuracies),
        "std_accuracy": np.std(accuracies)
    }

In [45]:
ablation_results = []

ablation_results.append(
    evaluate_ablation_policy(
        policy_name="Full (D,C,P)",
        policy_to_use=policy_full,
        state_function=state_full
    )
)

ablation_results.append(
    evaluate_ablation_policy(
        policy_name="DP only",
        policy_to_use=policy_dp,
        state_function=state_dp
    )
)

ablation_results.append(
    evaluate_ablation_policy(
        policy_name="D only",
        policy_to_use=policy_d,
        state_function=state_d
    )
)

ablation_results_df = pd.DataFrame(ablation_results)
print(ablation_results_df)

         policy  mean_reward  std_reward  mean_accuracy  std_accuracy
0  Full (D,C,P)     0.161748    0.217766       0.486133      0.150881
1       DP only     0.158939    0.216366       0.480133      0.148988
2        D only     0.160465    0.220104       0.482133      0.149562


In [47]:
import numpy as np
import pandas as pd
import math

for name in [
    "policy_full",
    "policy_dp",
    "policy_d",
    "state_full",
    "state_dp",
    "state_d",
    "evaluate_ablation_policy",
    "simulation_df",
]:
    print(name, name in globals())

policy_full True
policy_dp True
policy_d True
state_full True
state_dp True
state_d True
evaluate_ablation_policy True
simulation_df True


In [48]:
# ============================================================
# PHASE 16C — Statistical robustness of ablation results
# ============================================================

import numpy as np
import pandas as pd
from scipy import stats

# ------------------------------------------------------------
# 1. Evaluate one policy and KEEP episode-level results
# ------------------------------------------------------------

def evaluate_ablation_episodes(
    policy,
    state_function,
    n_episodes=300,
    steps_per_episode=50,
    seed=123
):
    rng = np.random.default_rng(seed)

    episode_rewards = []
    episode_accuracies = []

    for episode in range(n_episodes):

        total_reward = 0
        total_correct = 0

        # Random starting learner/text
        row_idx = rng.integers(0, len(simulation_df))
        row = simulation_df.iloc[row_idx]

        performance = float(row["Performance_P"])

        for step in range(steps_per_episode):

            # Build state according to the ablation configuration
            state = state_function(row, performance)

            # Choose learned action
            action = policy[state]

            # Environment interaction
            new_performance, reward, correct, p_before, p_after = environment_step(
                row=row,
                performance=performance,
                action=action,
                rng=rng
            )

            total_reward += reward
            total_correct += correct

            performance = new_performance

            # Keep same learner profile
            learner_rows = simulation_df[
                simulation_df["Learner_Profile"] ==
                row["Learner_Profile"]
            ]

            row = learner_rows.iloc[
                rng.integers(0, len(learner_rows))
            ]

        episode_rewards.append(
            total_reward / steps_per_episode
        )

        episode_accuracies.append(
            total_correct / steps_per_episode
        )

    return (
        np.array(episode_rewards),
        np.array(episode_accuracies)
    )

In [49]:
# ------------------------------------------------------------
# 2. State functions
# ------------------------------------------------------------

def state_full_eval(row, performance):
    return (
        int(row["D_level"]),
        int(row["C_level"]),
        performance_level(performance)
    )


def state_dp_eval(row, performance):
    return (
        int(row["D_level"]),
        performance_level(performance)
    )


def state_d_eval(row, performance):
    return int(row["D_level"])

In [53]:
for name in [
    "simulation_df",
    "performance_level",
    "environment_step",
    "environment_step_configurable",
    "probability_correct",
    "probability_correct_with_haptics",
    "haptic_effect",
    "np"
]:
    print(name, name in globals())

simulation_df True
performance_level True
environment_step False
environment_step_configurable True
probability_correct True
probability_correct_with_haptics True
haptic_effect True
np True


In [54]:
def environment_step(row, performance, action, rng):
    return environment_step_configurable(
        row=row,
        performance=performance,
        action=action,
        rng=rng,
        improvement_weight=0.3,
        costs={0: 0.00, 1: 0.02, 2: 0.05}
    )

In [56]:
def evaluate_ablation_episodes(
    policy,
    state_function,
    n_episodes=300,
    steps_per_episode=50,
    seed=2026
):
    rng = np.random.default_rng(seed)

    rewards = []
    accuracies = []

    for episode in range(n_episodes):

        row = simulation_df.iloc[
            rng.integers(0, len(simulation_df))
        ]

        performance = float(row["Performance_P"])

        total_reward = 0
        total_correct = 0

        for step in range(steps_per_episode):

            state = state_function(row, performance)

            if state not in policy:
                print("Missing state in policy:", state)
                print("Example policy keys:", list(policy.keys())[:10])
                raise KeyError(state)

            action = policy[state]

            new_performance, reward, correct, p_before, p_after = environment_step(
                row=row,
                performance=performance,
                action=action,
                rng=rng
            )

            total_reward += reward
            total_correct += correct

            learner_rows = simulation_df[
                simulation_df["Learner_Profile"] == row["Learner_Profile"]
            ]

            row = learner_rows.iloc[
                rng.integers(0, len(learner_rows))
            ]

            performance = new_performance

        rewards.append(total_reward / steps_per_episode)
        accuracies.append(total_correct / steps_per_episode)

    return np.mean(rewards), np.mean(accuracies)

In [57]:
print("Sample keys from policy_d:")
print(list(policy_d.keys())[:10])

print("One sample state from state_d_eval:")
sample_row = simulation_df.iloc[0]
print(state_d_eval(sample_row, float(sample_row["Performance_P"])))

Sample keys from policy_d:
[(0,), (1,), (2,)]
One sample state from state_d_eval:
2


In [58]:
def state_full_eval(row, performance):
    return (
        int(row["D_level"]),
        int(row["C_level"]),
        performance_level(performance)
    )

In [60]:
print("Sample keys from policy_d:")
print(list(policy_d.keys())[:10])

sample_row = simulation_df.iloc[0]
sample_perf = float(sample_row["Performance_P"])

print("Sample eval state:")
print(state_d_eval(sample_row, sample_perf))
print("Type:", type(state_d_eval(sample_row, sample_perf)))

Sample keys from policy_d:
[(0,), (1,), (2,)]
Sample eval state:
2
Type: <class 'int'>


In [61]:
def state_d_eval(row, performance):
    return (int(row["D_level"]),)

In [62]:
N_EPISODES = 300
STEPS = 50
TEST_SEED = 2026

full_reward, full_accuracy = evaluate_ablation_episodes(
    policy_full,
    state_full_eval,
    n_episodes=N_EPISODES,
    steps_per_episode=STEPS,
    seed=TEST_SEED
)

dp_reward, dp_accuracy = evaluate_ablation_episodes(
    policy_dp,
    state_dp_eval,
    n_episodes=N_EPISODES,
    steps_per_episode=STEPS,
    seed=TEST_SEED
)

d_reward, d_accuracy = evaluate_ablation_episodes(
    policy_d,
    state_d_eval,
    n_episodes=N_EPISODES,
    steps_per_episode=STEPS,
    seed=TEST_SEED
)

print("Full (D,C,P):", round(full_reward, 4), round(full_accuracy, 4))
print("Reduced (D,P):", round(dp_reward, 4), round(dp_accuracy, 4))
print("Difficulty only (D):", round(d_reward, 4), round(d_accuracy, 4))

Full (D,C,P): 0.227 0.5066
Reduced (D,P): 0.2218 0.4991
Difficulty only (D): 0.2252 0.5026


In [63]:
ablation_eval_df = pd.DataFrame({
    "model": ["Full (D,C,P)", "Reduced (D,P)", "Difficulty only (D)"],
    "mean_reward": [full_reward, dp_reward, d_reward],
    "mean_accuracy": [full_accuracy, dp_accuracy, d_accuracy]
})

print(ablation_eval_df.round(4))

                 model  mean_reward  mean_accuracy
0         Full (D,C,P)       0.2270         0.5066
1        Reduced (D,P)       0.2218         0.4991
2  Difficulty only (D)       0.2252         0.5026


In [64]:
import numpy as np
import pandas as pd

def environment_step(row, performance, action, rng):
    return environment_step_configurable(
        row=row,
        performance=performance,
        action=action,
        rng=rng,
        improvement_weight=0.3,
        costs={0: 0.00, 1: 0.02, 2: 0.05}
    )

In [65]:
def state_full_eval(row, performance):
    return (
        int(row["D_level"]),
        int(row["C_level"]),
        performance_level(performance)
    )

def state_dp_eval(row, performance):
    return (
        int(row["D_level"]),
        performance_level(performance)
    )

def state_d_eval(row, performance):
    return (int(row["D_level"]),)

In [66]:
def evaluate_ablation_episodes(
    policy,
    state_function,
    n_episodes=300,
    steps_per_episode=50,
    seed=2026
):
    rng = np.random.default_rng(seed)

    rewards = []
    accuracies = []

    for episode in range(n_episodes):
        row = simulation_df.iloc[rng.integers(0, len(simulation_df))]
        performance = float(row["Performance_P"])

        total_reward = 0
        total_correct = 0

        for step in range(steps_per_episode):
            state = state_function(row, performance)

            if state not in policy:
                print("Missing state:", state)
                print("Sample policy keys:", list(policy.keys())[:10])
                raise KeyError(state)

            action = policy[state]

            new_performance, reward, correct, _, _ = environment_step(
                row=row,
                performance=performance,
                action=action,
                rng=rng
            )

            total_reward += reward
            total_correct += correct

            learner_rows = simulation_df[
                simulation_df["Learner_Profile"] == row["Learner_Profile"]
            ]

            row = learner_rows.iloc[rng.integers(0, len(learner_rows))]
            performance = new_performance

        rewards.append(total_reward / steps_per_episode)
        accuracies.append(total_correct / steps_per_episode)

    return np.mean(rewards), np.mean(accuracies)

In [67]:
for name in [
    "simulation_df",
    "performance_level",
    "environment_step_configurable",
    "environment_step",
    "policy_full",
    "policy_dp",
    "policy_d",
    "state_full_eval",
    "state_dp_eval",
    "state_d_eval",
    "evaluate_ablation_episodes"
]:
    print(name, name in globals())

simulation_df True
performance_level True
environment_step_configurable True
environment_step True
policy_full True
policy_dp True
policy_d True
state_full_eval True
state_dp_eval True
state_d_eval True
evaluate_ablation_episodes True


In [68]:
# ------------------------------------------------------------
# 3. Episode-level evaluation
# ------------------------------------------------------------

N_EPISODES = 300
STEPS = 50
TEST_SEED = 2026

full_reward, full_accuracy = evaluate_ablation_episodes(
    policy_full,
    state_full_eval,
    n_episodes=N_EPISODES,
    steps_per_episode=STEPS,
    seed=TEST_SEED
)

dp_reward, dp_accuracy = evaluate_ablation_episodes(
    policy_dp,
    state_dp_eval,
    n_episodes=N_EPISODES,
    steps_per_episode=STEPS,
    seed=TEST_SEED
)

d_reward, d_accuracy = evaluate_ablation_episodes(
    policy_d,
    state_d_eval,
    n_episodes=N_EPISODES,
    steps_per_episode=STEPS,
    seed=TEST_SEED
)

print("Full (D,C,P):", round(full_reward, 4), round(full_accuracy, 4))
print("Reduced (D,P):", round(dp_reward, 4), round(dp_accuracy, 4))
print("Difficulty only (D):", round(d_reward, 4), round(d_accuracy, 4))

Full (D,C,P): 0.227 0.5066
Reduced (D,P): 0.2218 0.4991
Difficulty only (D): 0.2252 0.5026


In [69]:
# ------------------------------------------------------------
# 4. Descriptive results
# ------------------------------------------------------------

robustness_df = pd.DataFrame({
    "Model": [
        "Full (D,C,P)",
        "DP only",
        "D only"
    ],
    "Mean_reward": [
        full_reward,
        dp_reward,
        d_reward
    ],
    "Mean_accuracy": [
        full_accuracy,
        dp_accuracy,
        d_accuracy
    ]
})

print(robustness_df.round(4))

          Model  Mean_reward  Mean_accuracy
0  Full (D,C,P)       0.2270         0.5066
1       DP only       0.2218         0.4991
2        D only       0.2252         0.5026


In [71]:
# ------------------------------------------------------------
# 6. Reward comparisons
# ------------------------------------------------------------

print("This block only works after you create episode-level lists.")
print("You need:")
print("full_reward_list, dp_reward_list, d_reward_list")

This block only works after you create episode-level lists.
You need:
full_reward_list, dp_reward_list, d_reward_list


In [73]:
import numpy as np
from scipy import stats
import pandas as pd

def paired_comparison(full_values, reduced_values, comparison_name):
    full_values = np.array(full_values, dtype=float)
    reduced_values = np.array(reduced_values, dtype=float)

    difference = full_values - reduced_values

    t_stat, t_p = stats.ttest_rel(
        full_values,
        reduced_values
    )

    try:
        w_stat, w_p = stats.wilcoxon(
            full_values,
            reduced_values
        )
    except ValueError:
        w_stat = np.nan
        w_p = np.nan

    mean_difference = np.mean(difference)

    se = stats.sem(difference)

    ci_low, ci_high = stats.t.interval(
        0.95,
        df=len(difference) - 1,
        loc=mean_difference,
        scale=se
    )

    diff_sd = np.std(difference, ddof=1)

    if diff_sd == 0:
        cohens_dz = np.nan
    else:
        cohens_dz = mean_difference / diff_sd

    return {
        "Comparison": comparison_name,
        "Mean_difference": mean_difference,
        "CI_low": ci_low,
        "CI_high": ci_high,
        "t_statistic": t_stat,
        "t_p": t_p,
        "Wilcoxon_p": w_p,
        "Cohens_dz": cohens_dz
    }

In [76]:
import numpy as np
import pandas as pd
from scipy import stats

full_reward_list = np.array(full_reward, dtype=float)
dp_reward_list = np.array(dp_reward, dtype=float)
d_reward_list = np.array(d_reward, dtype=float)

def paired_comparison(full_values, reduced_values, comparison_name):
    full_values = np.array(full_values, dtype=float)
    reduced_values = np.array(reduced_values, dtype=float)

    difference = full_values - reduced_values

    t_stat, t_p = stats.ttest_rel(
        full_values,
        reduced_values
    )

    try:
        _, w_p = stats.wilcoxon(
            full_values,
            reduced_values
        )
    except ValueError:
        w_p = np.nan

    mean_difference = np.mean(difference)
    se = stats.sem(difference)

    ci_low, ci_high = stats.t.interval(
        0.95,
        df=len(difference) - 1,
        loc=mean_difference,
        scale=se
    )

    diff_sd = np.std(difference, ddof=1)

    if diff_sd == 0:
        cohens_dz = np.nan
    else:
        cohens_dz = mean_difference / diff_sd

    return {
        "Comparison": comparison_name,
        "Mean_difference": mean_difference,
        "CI_low": ci_low,
        "CI_high": ci_high,
        "t_statistic": t_stat,
        "t_p": t_p,
        "Wilcoxon_p": w_p,
        "Cohens_dz": cohens_dz
    }

In [79]:
import numpy as np
import pandas as pd
from scipy import stats

print("Shapes before testing:")
for name in ["full_reward_list", "dp_reward_list", "d_reward_list"]:
    value = globals()[name]
    arr = np.atleast_1d(np.asarray(value, dtype=float))
    print(name, "->", type(value), "shape:", arr.shape)

def paired_comparison_safe(full_values, reduced_values, comparison_name):
    full_values = np.atleast_1d(np.asarray(full_values, dtype=float))
    reduced_values = np.atleast_1d(np.asarray(reduced_values, dtype=float))

    if full_values.ndim != 1:
        full_values = full_values.ravel()

    if reduced_values.ndim != 1:
        reduced_values = reduced_values.ravel()

    if len(full_values) != len(reduced_values):
        raise ValueError(
            f"{comparison_name}: different lengths "
            f"{len(full_values)} vs {len(reduced_values)}"
        )

    difference = full_values - reduced_values
    n = len(difference)
    mean_difference = float(np.mean(difference))

    if n < 2:
        return {
            "Comparison": comparison_name,
            "N": n,
            "Mean_difference": mean_difference,
            "CI_low": np.nan,
            "CI_high": np.nan,
            "t_statistic": np.nan,
            "t_p": np.nan,
            "Wilcoxon_p": np.nan,
            "Cohens_dz": np.nan
        }

    t_stat, t_p = stats.ttest_rel(full_values, reduced_values)

    try:
        _, w_p = stats.wilcoxon(full_values, reduced_values)
    except Exception:
        w_p = np.nan

    diff_sd = np.std(difference, ddof=1)

    if diff_sd == 0 or np.isnan(diff_sd):
        cohens_dz = np.nan
    else:
        cohens_dz = mean_difference / diff_sd

    se = stats.sem(difference)
    ci_low, ci_high = stats.t.interval(
        0.95,
        df=n - 1,
        loc=mean_difference,
        scale=se
    )

    return {
        "Comparison": comparison_name,
        "N": n,
        "Mean_difference": mean_difference,
        "CI_low": ci_low,
        "CI_high": ci_high,
        "t_statistic": t_stat,
        "t_p": t_p,
        "Wilcoxon_p": w_p,
        "Cohens_dz": cohens_dz
    }

reward_tests = pd.DataFrame([
    paired_comparison_safe(full_reward_list, dp_reward_list, "Full vs DP"),
    paired_comparison_safe(full_reward_list, d_reward_list, "Full vs D")
])

print(reward_tests.round(5))

Shapes before testing:
full_reward_list -> <class 'numpy.ndarray'> shape: (1,)
dp_reward_list -> <class 'numpy.ndarray'> shape: (1,)
d_reward_list -> <class 'numpy.ndarray'> shape: (1,)
   Comparison  N  Mean_difference  CI_low  CI_high  t_statistic  t_p  \
0  Full vs DP  1          0.00520     NaN      NaN          NaN  NaN   
1   Full vs D  1          0.00179     NaN      NaN          NaN  NaN   

   Wilcoxon_p  Cohens_dz  
0         NaN        NaN  
1         NaN        NaN  


In [81]:
# ------------------------------------------------------------
# 6. Reward comparisons
# ------------------------------------------------------------

import numpy as np
import pandas as pd

reward_tests = pd.DataFrame([

    paired_comparison_safe(
        np.atleast_1d(full_reward),
        np.atleast_1d(dp_reward),
        "Full vs DP"
    ),

    paired_comparison_safe(
        np.atleast_1d(full_reward),
        np.atleast_1d(d_reward),
        "Full vs D"
    )

])

print("REWARD COMPARISONS")
print(reward_tests.round(5))

REWARD COMPARISONS
   Comparison  N  Mean_difference  CI_low  CI_high  t_statistic  t_p  \
0  Full vs DP  1          0.00520     NaN      NaN          NaN  NaN   
1   Full vs D  1          0.00179     NaN      NaN          NaN  NaN   

   Wilcoxon_p  Cohens_dz  
0         NaN        NaN  
1         NaN        NaN  


In [82]:
# ------------------------------------------------------------
# 7. Accuracy comparisons
# ------------------------------------------------------------

import numpy as np
import pandas as pd

accuracy_tests = pd.DataFrame([

    paired_comparison_safe(
        np.atleast_1d(full_accuracy),
        np.atleast_1d(dp_accuracy),
        "Full vs DP"
    ),

    paired_comparison_safe(
        np.atleast_1d(full_accuracy),
        np.atleast_1d(d_accuracy),
        "Full vs D"
    )

])

print("ACCURACY COMPARISONS")
print(accuracy_tests.round(5))

ACCURACY COMPARISONS
   Comparison  N  Mean_difference  CI_low  CI_high  t_statistic  t_p  \
0  Full vs DP  1          0.00753     NaN      NaN          NaN  NaN   
1   Full vs D  1          0.00400     NaN      NaN          NaN  NaN   

   Wilcoxon_p  Cohens_dz  
0         NaN        NaN  
1         NaN        NaN  


In [85]:
import numpy as np

def evaluate_episode_level(
    policy_to_use,
    state_function,
    n_episodes=300,
    steps_per_episode=50,
    seed=2026
):
    rng = np.random.default_rng(seed)

    episode_rewards = []
    episode_accuracies = []

    for episode in range(n_episodes):

        total_reward = 0.0
        total_correct = 0

        row = simulation_df.iloc[
            rng.integers(0, len(simulation_df))
        ]

        performance = float(
            row["Performance_P"]
        )

        for step in range(steps_per_episode):

            state = state_function(
                row,
                performance
            )

            action = int(
                policy_to_use[state]
            )

            (
                new_performance,
                reward,
                correct,
                _,
                _
            ) = environment_step(
                row=row,
                performance=performance,
                action=action,
                rng=rng
            )

            total_reward += reward
            total_correct += correct

            performance = new_performance

            learner_rows = simulation_df[
                simulation_df["Learner_Profile"]
                == row["Learner_Profile"]
            ]

            row = learner_rows.iloc[
                rng.integers(
                    0,
                    len(learner_rows)
                )
            ]

        episode_rewards.append(
            total_reward / steps_per_episode
        )

        episode_accuracies.append(
            total_correct / steps_per_episode
        )

    return (
        np.asarray(episode_rewards),
        np.asarray(episode_accuracies)
    )

In [86]:
N_EPISODES = 300
STEPS = 50
TEST_SEED = 2026

full_reward_list, full_accuracy_list = evaluate_episode_level(
    policy_to_use=policy_full,
    state_function=state_full_eval,
    n_episodes=N_EPISODES,
    steps_per_episode=STEPS,
    seed=TEST_SEED
)

dp_reward_list, dp_accuracy_list = evaluate_episode_level(
    policy_to_use=policy_dp,
    state_function=state_dp_eval,
    n_episodes=N_EPISODES,
    steps_per_episode=STEPS,
    seed=TEST_SEED
)

d_reward_list, d_accuracy_list = evaluate_episode_level(
    policy_to_use=policy_d,
    state_function=state_d_eval,
    n_episodes=N_EPISODES,
    steps_per_episode=STEPS,
    seed=TEST_SEED
)

print(full_reward_list.shape, full_accuracy_list.shape)
print(dp_reward_list.shape, dp_accuracy_list.shape)
print(d_reward_list.shape, d_accuracy_list.shape)

(300,) (300,)
(300,) (300,)
(300,) (300,)


In [88]:
def state_full_test(row, performance):
    return (
        int(row["D_level"]),
        int(row["C_level"]),
        performance_level(performance)
    )


def state_dp_test(row, performance):
    return (
        int(row["D_level"]),
        performance_level(performance)
    )


def state_d_test(row, performance):
    return (
        int(row["D_level"]),
    )

In [89]:
N_EPISODES = 300
STEPS = 50
TEST_SEED = 2026

full_reward_list, full_accuracy_list = evaluate_episode_level(
    policy_to_use=policy_full,
    state_function=state_full_test,
    n_episodes=N_EPISODES,
    steps_per_episode=STEPS,
    seed=TEST_SEED
)

dp_reward_list, dp_accuracy_list = evaluate_episode_level(
    policy_to_use=policy_dp,
    state_function=state_dp_test,
    n_episodes=N_EPISODES,
    steps_per_episode=STEPS,
    seed=TEST_SEED
)

d_reward_list, d_accuracy_list = evaluate_episode_level(
    policy_to_use=policy_d,
    state_function=state_d_test,
    n_episodes=N_EPISODES,
    steps_per_episode=STEPS,
    seed=TEST_SEED
)

print(full_reward_list.shape, full_accuracy_list.shape)
print(dp_reward_list.shape, dp_accuracy_list.shape)
print(d_reward_list.shape, d_accuracy_list.shape)

(300,) (300,)
(300,) (300,)
(300,) (300,)


In [90]:
robustness_df = pd.DataFrame({
    "Model": [
        "Full (D,C,P)",
        "DP only",
        "D only"
    ],
    "Mean_reward": [
        np.mean(full_reward_list),
        np.mean(dp_reward_list),
        np.mean(d_reward_list)
    ],
    "SD_reward": [
        np.std(full_reward_list, ddof=1),
        np.std(dp_reward_list, ddof=1),
        np.std(d_reward_list, ddof=1)
    ],
    "Mean_accuracy": [
        np.mean(full_accuracy_list),
        np.mean(dp_accuracy_list),
        np.mean(d_accuracy_list)
    ],
    "SD_accuracy": [
        np.std(full_accuracy_list, ddof=1),
        np.std(dp_accuracy_list, ddof=1),
        np.std(d_accuracy_list, ddof=1)
    ]
})

print(robustness_df.round(4))

          Model  Mean_reward  SD_reward  Mean_accuracy  SD_accuracy
0  Full (D,C,P)       0.2270     0.2042         0.5066       0.1389
1       DP only       0.2218     0.2052         0.4991       0.1393
2        D only       0.2252     0.2058         0.5026       0.1383


In [91]:
for name in [
    "full_reward_list",
    "full_accuracy_list",
    "dp_reward_list",
    "dp_accuracy_list",
    "d_reward_list",
    "d_accuracy_list"
]:
    print(name, name in globals())

print("\nShapes:")

print(
    "Full:",
    np.asarray(full_reward_list).shape,
    np.asarray(full_accuracy_list).shape
)

print(
    "DP:",
    np.asarray(dp_reward_list).shape,
    np.asarray(dp_accuracy_list).shape
)

print(
    "D:",
    np.asarray(d_reward_list).shape,
    np.asarray(d_accuracy_list).shape
)

full_reward_list True
full_accuracy_list True
dp_reward_list True
dp_accuracy_list True
d_reward_list True
d_accuracy_list True

Shapes:
Full: (300,) (300,)
DP: (300,) (300,)
D: (300,) (300,)


In [92]:
import numpy as np
import pandas as pd

full_reward_list = np.asarray(full_reward_list, dtype=float)
dp_reward_list = np.asarray(dp_reward_list, dtype=float)
d_reward_list = np.asarray(d_reward_list, dtype=float)

full_accuracy_list = np.asarray(full_accuracy_list, dtype=float)
dp_accuracy_list = np.asarray(dp_accuracy_list, dtype=float)
d_accuracy_list = np.asarray(d_accuracy_list, dtype=float)

robustness_df = pd.DataFrame({
    "Model": [
        "Full (D,C,P)",
        "DP only",
        "D only"
    ],
    "Mean_Reward": [
        full_reward_list.mean(),
        dp_reward_list.mean(),
        d_reward_list.mean()
    ],
    "SD_Reward": [
        full_reward_list.std(ddof=1),
        dp_reward_list.std(ddof=1),
        d_reward_list.std(ddof=1)
    ],
    "Mean_Accuracy": [
        full_accuracy_list.mean(),
        dp_accuracy_list.mean(),
        d_accuracy_list.mean()
    ],
    "SD_Accuracy": [
        full_accuracy_list.std(ddof=1),
        dp_accuracy_list.std(ddof=1),
        d_accuracy_list.std(ddof=1)
    ]
})

print(robustness_df.round(4))

          Model  Mean_Reward  SD_Reward  Mean_Accuracy  SD_Accuracy
0  Full (D,C,P)       0.2270     0.2042         0.5066       0.1389
1       DP only       0.2218     0.2052         0.4991       0.1393
2        D only       0.2252     0.2058         0.5026       0.1383


In [93]:
def paired_comparison(
    full_values,
    reduced_values,
    name
):
    full_values = np.asarray(
        full_values,
        dtype=float
    )

    reduced_values = np.asarray(
        reduced_values,
        dtype=float
    )

    difference = (
        full_values
        - reduced_values
    )

    n = len(difference)

    mean_diff = difference.mean()
    sd_diff = difference.std(ddof=1)

    se_diff = (
        sd_diff / np.sqrt(n)
    )

    ci_low = (
        mean_diff
        - stats.t.ppf(
            0.975,
            df=n - 1
        ) * se_diff
    )

    ci_high = (
        mean_diff
        + stats.t.ppf(
            0.975,
            df=n - 1
        ) * se_diff
    )

    t_stat, t_p = stats.ttest_rel(
        full_values,
        reduced_values
    )

    try:
        _, wilcoxon_p = stats.wilcoxon(
            full_values,
            reduced_values
        )
    except ValueError:
        wilcoxon_p = np.nan

    cohens_dz = (
        mean_diff / sd_diff
        if sd_diff > 0
        else np.nan
    )

    return {
        "Comparison": name,
        "N": n,
        "Mean_difference": mean_diff,
        "CI_low": ci_low,
        "CI_high": ci_high,
        "t_statistic": t_stat,
        "t_p": t_p,
        "Wilcoxon_p": wilcoxon_p,
        "Cohens_dz": cohens_dz
    }

In [94]:
reward_tests = pd.DataFrame([
    paired_comparison(
        full_reward_list,
        dp_reward_list,
        "Full vs DP"
    ),
    paired_comparison(
        full_reward_list,
        d_reward_list,
        "Full vs D"
    )
])

print("REWARD COMPARISONS")
print(reward_tests.round(5))

REWARD COMPARISONS
   Comparison    N  Mean_difference   CI_low  CI_high  t_statistic      t_p  \
0  Full vs DP  300          0.00520 -0.00122  0.01161      1.59440  0.11190   
1   Full vs D  300          0.00179 -0.00494  0.00853      0.52397  0.60069   

   Wilcoxon_p  Cohens_dz  
0     0.53103    0.09205  
1     0.86796    0.03025  


In [95]:
accuracy_tests = pd.DataFrame([
    paired_comparison(
        full_accuracy_list,
        dp_accuracy_list,
        "Full vs DP"
    ),
    paired_comparison(
        full_accuracy_list,
        d_accuracy_list,
        "Full vs D"
    )
])

print("ACCURACY COMPARISONS")
print(accuracy_tests.round(5))

ACCURACY COMPARISONS
   Comparison    N  Mean_difference   CI_low  CI_high  t_statistic      t_p  \
0  Full vs DP  300          0.00753  0.00336  0.01170      3.55599  0.00044   
1   Full vs D  300          0.00400 -0.00034  0.00834      1.81432  0.07063   

   Wilcoxon_p  Cohens_dz  
0     0.00126    0.20531  
1     0.06820    0.10475  
